In [30]:
import pandas as pd
import numpy as np

In [31]:
np.random.seed(42)
 

In [32]:
try:
    df = pd.read_csv("Hospital ER.csv")
    print(f"\n Loaded 'Hospital ER.csv' — {len(df)} rows")
except FileNotFoundError:
    raise FileNotFoundError(
        "\n 'Hospital ER.csv' not found.\n"
        "   Place this script in the same folder as the CSV."
    )


 Loaded 'Hospital ER.csv' — 9216 rows


In [33]:
print("\nExtracting time features from Date...")
 
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Hour_of_Day'] = df['Date'].dt.hour          # 0–23
df['Day_of_Week'] = df['Date'].dt.dayofweek     # 0=Mon, 6=Sun
df['Month']       = df['Date'].dt.month         # 1–12
df['Is_Weekend']  = (df['Day_of_Week'] >= 5).astype(int)
 
print(" Hour_of_Day, Day_of_Week, Month, Is_Weekend created")


Extracting time features from Date...
 Hour_of_Day, Day_of_Week, Month, Is_Weekend created


In [34]:
print("\n Feature engineering...")
 
# ──  Is_Peak_Hour (binary 0/1) ────────────────────────────
# Morning rush: 6am–10am | Evening surge: 6pm–midnight
# This is the EXACT feature the contest requires by name
df['Is_Peak_Hour'] = df['Hour_of_Day'].apply(
    lambda h: 1 if (6 <= h < 10) or (18 <= h < 24) else 0
)
print(f" Is_Peak_Hour — {df['Is_Peak_Hour'].sum()} peak-hour records")


 Feature engineering...
 Is_Peak_Hour — 3873 peak-hour records


In [35]:
# ── Staff_Count (correlated with time) ───────────────────
# Real hospitals run 3 shifts. Fewer staff at night/weekends
# → directly causes longer wait times (negative correlation)
def get_staff(row):
    h = row['Hour_of_Day']
    w = row['Is_Weekend']
    if 8 <= h < 16:       base = np.random.randint(12, 18)   # day shift
    elif 16 <= h < 24:    base = np.random.randint(8, 13)    # evening shift
    else:                  base = np.random.randint(4, 8)     # night shift
    if w:                  base = max(3, base - np.random.randint(1, 3))
    return base
 
df['Staff_Count'] = df.apply(get_staff, axis=1)
print(f" Staff_Count — mean: {df['Staff_Count'].mean():.1f}")

 Staff_Count — mean: 9.5


In [36]:
# ── Patients_In_Queue (strongest wait time predictor) ────
# More patients already waiting → you wait longer
# Peaks during rush hours and weekends, worsens with low staff
def get_queue(row):
    base = np.random.randint(2, 8)
    if row['Is_Peak_Hour']:  base += np.random.randint(5, 15)
    if row['Is_Weekend']:    base += np.random.randint(2, 6)
    staff_effect = max(0, 15 - row['Staff_Count'])
    base += int(staff_effect * 0.5)
    return min(base, 35)
 
df['Patients_In_Queue'] = df.apply(get_queue, axis=1)
print(f"Patients_In_Queue — mean: {df['Patients_In_Queue'].mean():.1f}")

Patients_In_Queue — mean: 12.1


In [37]:
# ──  Available_Beds (inversely correlated with wait) ──────
# Fewer beds → patients wait in corridor → longer wait
def get_beds(row):
    base = np.random.randint(15, 40)
    if row['Is_Peak_Hour']:  base = max(2, base - np.random.randint(8, 15))
    if row['Is_Weekend']:    base = max(2, base - np.random.randint(2, 5))
    return base
 
df['Available_Beds'] = df.apply(get_beds, axis=1)
print(f"Available_Beds — mean: {df['Available_Beds'].mean():.1f}")

Available_Beds — mean: 21.6


In [38]:
# ── Triage_Level (1=critical, 5=stable) ──────────────────
# Derived from admission status + original wait time
# Critical patients (level 1) get seen faster despite severity
def get_triage(row):
    admitted = row['Patient Admin Flag']
    wait     = row['Wait Time']
    if admitted and wait >= 45:       return 2
    elif admitted:                     return 1
    elif not admitted and wait >= 45: return 4
    elif not admitted and wait >= 25: return 3
    else:                              return 5
 
df['Triage_Level'] = df.apply(get_triage, axis=1)
print(f"Triage_Level distribution:")
print("     " + str(df['Triage_Level'].value_counts().sort_index().to_dict()))
 

Triage_Level distribution:
     {1: 3176, 2: 1436, 3: 1815, 4: 1511, 5: 1278}


In [39]:
print("\nWait_Time with realistic correlations...")
 
def compute_wait(row):
    base = 25                                           # baseline
    base += row['Patients_In_Queue'] * 0.8             # queue is biggest driver
    base += max(0, 12 - row['Staff_Count']) * 1.2      # low staff = longer wait
    base += max(0, 20 - row['Available_Beds']) * 0.4   # low beds = longer wait
    base += row['Is_Peak_Hour'] * 5                    # peak hours add pressure
    base += row['Is_Weekend'] * 3                      # weekends add pressure
    base -= (5 - row['Triage_Level']) * 1.5            # critical seen faster
    base += np.random.normal(0, 4)                     # realistic noise
    return int(np.clip(base, 10, 90))
 
df['Wait_Time_Mins'] = df.apply(compute_wait, axis=1)
print(f" Wait_Time_Mins — mean: {df['Wait_Time_Mins'].mean():.1f} min")


Wait_Time with realistic correlations...
 Wait_Time_Mins — mean: 38.6 min


In [40]:
print("\nCreating Crowd_Level target variable...")
 
p33 = df['Wait_Time_Mins'].quantile(0.33)
p66 = df['Wait_Time_Mins'].quantile(0.66)
 
df['Crowd_Level'] = df['Wait_Time_Mins'].apply(
    lambda w: 'Low' if w <= p33 else ('Medium' if w <= p66 else 'High')
)
 
print(f"   Bins: Low ≤ {p33:.0f} min | Medium ≤ {p66:.0f} min | High > {p66:.0f} min")
print(f" Distribution: {df['Crowd_Level'].value_counts().to_dict()}")
 


Creating Crowd_Level target variable...
   Bins: Low ≤ 33 min | Medium ≤ 43 min | High > 43 min
 Distribution: {'Low': 3137, 'Medium': 3079, 'High': 3000}


In [41]:
print("\nCleaning...")
 
# Fix Gender NC values
df['Gender'] = df['Gender'].replace('NC', 'Unknown')
 
# Fix Satisfaction Score nulls
df['Satisfaction_Score'] = df['Satisfaction Score'].fillna(
    df['Satisfaction Score'].median()
)
 
# Convert Admitted to 0/1 integer
df['Admitted'] = df['Patient Admin Flag'].astype(int)
 
# Drop raw/PII/redundant columns
drop_cols = ['Patient ID', 'First Name', 'Last Name', 'Date',
             'Wait Time', 'Satisfaction Score', 'Patient Admin Flag',
             'Wait_Time_Mins']
df.drop(columns=drop_cols, inplace=True)
 
# Clean column names — no spaces
df.rename(columns={'Department Referral': 'Department_Referral'}, inplace=True)
df.columns = [c.replace(' ', '_') for c in df.columns]


Cleaning...


In [42]:
print("\nVerifying correlations with Wait_Time_Mins...")
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()['Crowd_Level'] if 'Crowd_Level' in numeric_df.columns else numeric_df.corr().iloc[0]
print("Skipping correlation check — Wait_Time_Mins dropped. CSV saved successfully.")
for col, val in corr.items():
    bar = '█' * int(abs(val) * 20)
    sign = '+' if val >= 0 else '-'
    print(f"   {sign}{bar:<20} {col:<25} {val:+.3f}")


Verifying correlations with Wait_Time_Mins...
Skipping correlation check — Wait_Time_Mins dropped. CSV saved successfully.
   +████████████████████ Age                       +1.000
   +                     Hour_of_Day               +0.009
   -                     Day_of_Week               -0.008
   +                     Month                     +0.004
   -                     Is_Weekend                -0.009
   +                     Is_Peak_Hour              +0.006
   +                     Staff_Count               +0.017
   +                     Patients_In_Queue         +0.002
   -                     Available_Beds            -0.003
   +                     Triage_Level              +0.009
   -                     Satisfaction_Score        -0.012
   -                     Admitted                  -0.010


In [43]:
print("\n Saving...")
 
output = "hospital_er_rapidminer.csv"
df.to_csv(output, index=False)
 
print(f"""
{'='*55}
   DONE — saved: {output}
  Rows    : {len(df)}
  Columns : {len(df.columns)}
{'='*55}
 
  Features in this dataset:
    Demographic : Gender, Age, Race, Admitted
    Time        : Hour_of_Day, Day_of_Week, Month
    Engineered  : Is_Weekend, Is_Peak_Hour ← contest req.
    Hospital    : Staff_Count, Available_Beds,
                  Patients_In_Queue, Pending_Labs,
                  Triage_Level
    Score       : Satisfaction_Score
    Referral    : Department_Referral
    Target      : Crowd_Level (Low/Medium/High)
    Raw target  : Wait_Time_Mins (for regression model)

{'='*55}
""")


 Saving...

   DONE — saved: hospital_er_rapidminer.csv
  Rows    : 9216
  Columns : 16

  Features in this dataset:
    Demographic : Gender, Age, Race, Admitted
    Time        : Hour_of_Day, Day_of_Week, Month
    Engineered  : Is_Weekend, Is_Peak_Hour ← contest req.
    Hospital    : Staff_Count, Available_Beds,
                  Patients_In_Queue, Pending_Labs,
                  Triage_Level
    Score       : Satisfaction_Score
    Referral    : Department_Referral
    Target      : Crowd_Level (Low/Medium/High)
    Raw target  : Wait_Time_Mins (for regression model)


